In [9]:
import pickle 

BASE_PATH = "C:/CLG LAB/5TH SEM/NLP/lab5/saved_data/ngram_models/"

for i in range(4): 
    i+=1
    with open(BASE_PATH+f"{i}gram_counts.pkl","br") as f:
        if i==1:
            uni_counts = pickle.load(f) 
        elif i==2:
            bi_counts = pickle.load(f)
        elif i==3:
            tri_counts = pickle.load(f)
        else:
            quad_counts = pickle.load(f)
        
total_unigrams = sum(uni_counts.values())

In [10]:
print(len(uni_counts), "unigrams")
print(len(bi_counts), "bigrams")
print(len(tri_counts), "trigrams")
print(len(quad_counts), "quadrigrams")

# Example: check first 5 quadrigrams
for i, (k, v) in enumerate(quad_counts.items()):
    print(k, v)
    if i == 4:
        break


153603 unigrams
785173 bigrams
1121260 trigrams
1231944 quadrigrams
('<s>', '<s>', '<s>', 'આ') 6779
('<s>', '<s>', 'આ', 'વીડિયો') 43
('<s>', 'આ', 'વીડિયો', 'જુઓ:') 4
('આ', 'વીડિયો', 'જુઓ:', 'ઊંઝા') 1
('વીડિયો', 'જુઓ:', 'ઊંઝા', 'માર્કેટયાર્ડ') 1


# Katx backoff model 

In [22]:
def katz_backoff_prob(w1, w2, w3, w4, d=0.5):
    quad = quad_counts.get((w1, w2, w3, w4), 0)
    tri = tri_counts.get((w1, w2, w3), 0)
    bi = bi_counts.get((w2, w3), 0)
    uni = uni_counts.get(w3, 0)
    
    # 4-gram
    if quad > 0 and tri > 0:
        return max((quad - d) / tri, 0)
    
    # 3-gram
    elif tri_counts.get((w2, w3, w4), 0) > 0 and bi > 0:
        return max((tri_counts[(w2, w3, w4)] - d) / bi, 0)
    
    # 2-gram
    elif bi_counts.get((w3, w4), 0) > 0 and uni > 0:
        return max((bi_counts[(w3, w4)] - d) / uni, 0)
    
    # unigram
    elif uni_counts.get(w4, 0) > 0:
        return uni_counts[w4] / total_unigrams
    
    return 1 / total_unigrams


# load validation sentences 

In [23]:
# Read validation sentences
file_path = "C:/CLG LAB/5TH SEM/NLP/lab5/saved_data/validation_set_1lakh.txt"

# Read only first 10 sentences
validation_sentences = []
with open(file_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i == 10:  # stop after 10 lines
            break
        line = line.strip()
        if line:  # skip empty lines
            validation_sentences.append(line)

print("Loaded", len(validation_sentences), "validation sentences:")
for s in validation_sentences:
    print("-", s)


Loaded 10 validation sentences:
- ૭પ) (નિવૃત જીઇબી એન્જીનીયર) તે હીરેશનભાઇ, ચેતનભાઇ ના પિતાશ્રીનું તા.
- ભાવનગરનાં નિલમબાગ પોલીસ મથકમાં અધેવાડામાં રહેતા ઇન્દ્વજીતસિંહ ઉર્ફે ઇનો વિક્રમસિંહ ગોહિલ કાચા કામના કેદી તરીકે ભાવનગર જેલમાં સજા ભોગી રહ્યો હતો.
- ઓટોરિક્ષા ચાલકોને નવી ઓળખ મળી ગઈ છે.
- સારવાર બાદ તેઓ સ્વસ્થ થયા હતા અને તેમને રજા પણ આપવામાં આવી હતી.
- રસ્તામાં ખાડાઓ પડેલા જોવા મળે છે અને ભુવો પડવાના કિસ્સાઓ પણ વધી જતા હોય છે.
- ગ્લો મોડ એ જૂતાની અંદરના ભાગમાં ખાસ પ્રદાન કરેલા બટન દ્વારા સ્વિચ કરવામાં આવે છે.
- આ ટ્રેનિંગ કેન્દ્રસરકારના આરોગ્ય કર્મચારી અને વર્લ્ડ હૅલ્થ ઑર્ગેનાઈઝેન (હુ)ના માર્ગદર્શન હેઠળ આપવામાં આવી રહી હોવાનું પાલિકાના એડિશનલ કમિશનર સુરેશ કાકાણીએ જણાવ્યું હતું.
- વહેલી સવારથી જ ઠંડા પવન ફ્ૂંકાતા રહીશો મુશ્કેલીમાં મુકાયા હતા.
- દર્શનાબેન પંડયા, પ્રીતીબેન દોશી, પીનાબેન કોટક, વોર્ડ નં.
- સેન્ટ્રિપ્ટલ કારકિર્દીની રેખા (નિયંત્રણ કેન્દ્રમાં પ્રમોશન અને મેનેજમેન્ટ અને નિર્ણય લેવાની પ્રક્રિયામાં જોડાયા)


In [24]:
def preprocess_sentence(sentence):
    tokens = sentence.lower().split()
    tokens = ['<s>', '<s>', '<s>'] + tokens + ['</s>']
    return tokens


# compute probability for one sentence

In [25]:
import math

def sentence_probability(sentence, katz_func):
    tokens = preprocess_sentence(sentence)
    prob = 1.0
    log_prob = 0.0
    
    for i in range(3, len(tokens)):
        w1, w2, w3, w4 = tokens[i-3], tokens[i-2], tokens[i-1], tokens[i]
        p = katz_backoff_prob(w1, w2, w3, w4)
        prob *= p
        log_prob += math.log10(p) if p > 0 else math.log10(1e-10)
    
    return prob, log_prob


# probability for 10 validation sentences 

In [26]:
sentence_probs = {}

for sent in validation_sentences:
    prob, log_prob = sentence_probability(sent, katz_backoff_prob)
    sentence_probs[sent] = {"probability": prob, "log_probability": log_prob}

# Display results
for sent, vals in sentence_probs.items():
    print(f"\nSentence: {sent}")
    print(f"Probability: {vals['probability']:.6e}")
    print(f"Log Probability: {vals['log_probability']:.4f}")



Sentence: ૭પ) (નિવૃત જીઇબી એન્જીનીયર) તે હીરેશનભાઇ, ચેતનભાઇ ના પિતાશ્રીનું તા.
Probability: 3.121717e-62
Log Probability: -61.5056

Sentence: ભાવનગરનાં નિલમબાગ પોલીસ મથકમાં અધેવાડામાં રહેતા ઇન્દ્વજીતસિંહ ઉર્ફે ઇનો વિક્રમસિંહ ગોહિલ કાચા કામના કેદી તરીકે ભાવનગર જેલમાં સજા ભોગી રહ્યો હતો.
Probability: 1.007294e-129
Log Probability: -128.9968

Sentence: ઓટોરિક્ષા ચાલકોને નવી ઓળખ મળી ગઈ છે.
Probability: 3.879969e-38
Log Probability: -37.4112

Sentence: સારવાર બાદ તેઓ સ્વસ્થ થયા હતા અને તેમને રજા પણ આપવામાં આવી હતી.
Probability: 4.807440e-60
Log Probability: -59.3181

Sentence: રસ્તામાં ખાડાઓ પડેલા જોવા મળે છે અને ભુવો પડવાના કિસ્સાઓ પણ વધી જતા હોય છે.
Probability: 5.474969e-72
Log Probability: -71.2616

Sentence: ગ્લો મોડ એ જૂતાની અંદરના ભાગમાં ખાસ પ્રદાન કરેલા બટન દ્વારા સ્વિચ કરવામાં આવે છે.
Probability: 5.299708e-87
Log Probability: -86.2757

Sentence: આ ટ્રેનિંગ કેન્દ્રસરકારના આરોગ્ય કર્મચારી અને વર્લ્ડ હૅલ્થ ઑર્ગેનાઈઝેન (હુ)ના માર્ગદર્શન હેઠળ આપવામાં આવી રહી હોવાનું પાલિકાના એડિશનલ કમ

# saving into csv file 

In [27]:
import pandas as pd

df = pd.DataFrame([
    {"Sentence": s, 
     "Probability": vals["probability"], 
     "Log Probability": vals["log_probability"]}
    for s, vals in sentence_probs.items()
])

df.to_csv("validation_sentence_probabilities.csv", index=False)
print("Saved to validation_sentence_probabilities.csv")


Saved to validation_sentence_probabilities.csv


In [29]:
df = pd.read_csv("validation_sentence_probabilities.csv")

df

,Sentence,Probability,Log Probability
0,"૭પ) (નિવૃત જીઇબી એન્જીનીયર) તે હીરેશનભાઇ, ચેતન...",3.121717e-62,-61.505606
1,ભાવનગરનાં નિલમબાગ પોલીસ મથકમાં અધેવાડામાં રહેત...,1.007294e-129,-128.996844
2,ઓટોરિક્ષા ચાલકોને નવી ઓળખ મળી ગઈ છે.,3.879969e-38,-37.411172
3,સારવાર બાદ તેઓ સ્વસ્થ થયા હતા અને તેમને રજા પણ...,4.807440e-60,-59.318086
4,રસ્તામાં ખાડાઓ પડેલા જોવા મળે છે અને ભુવો પડવા...,5.474969e-72,-71.261618
5,ગ્લો મોડ એ જૂતાની અંદરના ભાગમાં ખાસ પ્રદાન કરે...,5.299708e-87,-86.275748
6,આ ટ્રેનિંગ કેન્દ્રસરકારના આરોગ્ય કર્મચારી અને ...,6.887838e-137,-136.161917
7,વહેલી સવારથી જ ઠંડા પવન ફ્ૂંકાતા રહીશો મુશ્કેલ...,7.055615e-46,-45.151465
8,"દર્શનાબેન પંડયા, પ્રીતીબેન દોશી, પીનાબેન કોટક,...",7.151637e-50,-49.145595
9,સેન્ટ્રિપ્ટલ કારકિર્દીની રેખા (નિયંત્રણ કેન્દ્...,3.496127e-81,-80.456413


# Kneser-Ney smoothing 

# funtion for continuation counts 

In [30]:
from collections import defaultdict

# Continuation counts for Kneser-Ney
continuation_count_uni = defaultdict(int)
continuation_count_bi = defaultdict(int)
continuation_count_tri = defaultdict(int)

# Each word's number of distinct left contexts
for (w1, w2) in bi_counts.keys():
    continuation_count_uni[w2] += 1

for (w1, w2, w3) in tri_counts.keys():
    continuation_count_bi[(w2, w3)] += 1

for (w1, w2, w3, w4) in quad_counts.keys():
    continuation_count_tri[(w2, w3, w4)] += 1

print("Continuation counts computed.")


Continuation counts computed.


# kneser-ney probability

In [34]:
def kneser_ney_prob(w1, w2, w3, w4, d=0.75):
    def safe_div(num, den):
        return num / den if den > 0 else 0

    # 4-gram counts
    c4 = quad_counts.get((w1, w2, w3, w4), 0)
    c3 = tri_counts.get((w1, w2, w3), 0)

    # Number of unique words following (w1, w2, w3)
    num_followers = len([1 for (a, b, c, d4) in quad_counts if (a, b, c) == (w1, w2, w3)])
    lambda3 = (d * num_followers / c3) if c3 > 0 else 1

    # Backoff to trigram level
    p_continuation3 = kneser_ney_trigram_prob(w2, w3, w4, d)

    return safe_div(max(c4 - d, 0), c3) + lambda3 * p_continuation3


def kneser_ney_trigram_prob(w2, w3, w4, d=0.75):
    c3 = tri_counts.get((w2, w3, w4), 0)
    c2 = bi_counts.get((w2, w3), 0)

    num_followers = len([1 for (a, b, c) in tri_counts if (a, b) == (w2, w3)])
    lambda2 = (d * num_followers / c2) if c2 > 0 else 1

    p_continuation2 = kneser_ney_bigram_prob(w3, w4, d)
    return (max(c3 - d, 0) / c2 if c2 > 0 else 0) + lambda2 * p_continuation2


def kneser_ney_bigram_prob(w3, w4, d=0.75):
    c2 = bi_counts.get((w3, w4), 0)
    c1 = uni_counts.get(w3, 0)

    num_followers = len([1 for (a, b) in bi_counts if a == w3])
    lambda1 = (d * num_followers / c1) if c1 > 0 else 1

    # Continuation probability: how many bigrams end with w4
    p_continuation1 = continuation_count_uni[w4] / len(bi_counts)

    return (max(c2 - d, 0) / c1 if c1 > 0 else 0) + lambda1 * p_continuation1


In [35]:
import math

def sentence_probability(sentence, kn_func):
    tokens = preprocess_sentence(sentence)
    prob = 1.0
    log_prob = 0.0
    
    for i in range(3, len(tokens)):
        w1, w2, w3, w4 = tokens[i-3], tokens[i-2], tokens[i-1], tokens[i]
        p = kn_func(w1, w2, w3, w4)
        prob *= p
        log_prob += math.log10(p) if p > 0 else math.log10(1e-10)
    
    return prob, log_prob


In [36]:
sentence_probs_kn = {}

for sent in validation_sentences:
    prob, log_prob = sentence_probability(sent, kneser_ney_prob)
    sentence_probs_kn[sent] = {"probability": prob, "log_probability": log_prob}

for sent, vals in sentence_probs_kn.items():
    print(f"\nSentence: {sent}")
    print(f"Probability: {vals['probability']:.6e}")
    print(f"Log Probability: {vals['log_probability']:.4f}")



Sentence: ૭પ) (નિવૃત જીઇબી એન્જીનીયર) તે હીરેશનભાઇ, ચેતનભાઇ ના પિતાશ્રીનું તા.
Probability: 0.000000e+00
Log Probability: -65.9949

Sentence: ભાવનગરનાં નિલમબાગ પોલીસ મથકમાં અધેવાડામાં રહેતા ઇન્દ્વજીતસિંહ ઉર્ફે ઇનો વિક્રમસિંહ ગોહિલ કાચા કામના કેદી તરીકે ભાવનગર જેલમાં સજા ભોગી રહ્યો હતો.
Probability: 0.000000e+00
Log Probability: -104.2376

Sentence: ઓટોરિક્ષા ચાલકોને નવી ઓળખ મળી ગઈ છે.
Probability: 0.000000e+00
Log Probability: -30.0160

Sentence: સારવાર બાદ તેઓ સ્વસ્થ થયા હતા અને તેમને રજા પણ આપવામાં આવી હતી.
Probability: 1.221741e-36
Log Probability: -35.9130

Sentence: રસ્તામાં ખાડાઓ પડેલા જોવા મળે છે અને ભુવો પડવાના કિસ્સાઓ પણ વધી જતા હોય છે.
Probability: 4.632688e-50
Log Probability: -49.3342

Sentence: ગ્લો મોડ એ જૂતાની અંદરના ભાગમાં ખાસ પ્રદાન કરેલા બટન દ્વારા સ્વિચ કરવામાં આવે છે.
Probability: 2.837038e-57
Log Probability: -56.5471

Sentence: આ ટ્રેનિંગ કેન્દ્રસરકારના આરોગ્ય કર્મચારી અને વર્લ્ડ હૅલ્થ ઑર્ગેનાઈઝેન (હુ)ના માર્ગદર્શન હેઠળ આપવામાં આવી રહી હોવાનું પાલિકાના એડિશનલ કમિ

# save to csv

In [37]:
import pandas as pd

df = pd.DataFrame([
    {"Sentence": s, 
     "Probability": vals["probability"], 
     "Log Probability": vals["log_probability"]}
    for s, vals in sentence_probs_kn.items()
])

df.to_csv("validation_sentence_probabilities_kneser_ney.csv", index=False)
print("Saved to validation_sentence_probabilities_kneser_ney.csv")


Saved to validation_sentence_probabilities_kneser_ney.csv


In [38]:
kneser_ney_probs = pd.read_csv("validation_sentence_probabilities_kneser_ney.csv")
kneser_ney_probs

,Sentence,Probability,Log Probability
0,"૭પ) (નિવૃત જીઇબી એન્જીનીયર) તે હીરેશનભાઇ, ચેતન...",0.000000e+00,-65.994879
1,ભાવનગરનાં નિલમબાગ પોલીસ મથકમાં અધેવાડામાં રહેત...,0.000000e+00,-104.237565
2,ઓટોરિક્ષા ચાલકોને નવી ઓળખ મળી ગઈ છે.,0.000000e+00,-30.016038
3,સારવાર બાદ તેઓ સ્વસ્થ થયા હતા અને તેમને રજા પણ...,1.221741e-36,-35.913021
4,રસ્તામાં ખાડાઓ પડેલા જોવા મળે છે અને ભુવો પડવા...,4.632688e-50,-49.334167
5,ગ્લો મોડ એ જૂતાની અંદરના ભાગમાં ખાસ પ્રદાન કરે...,2.837038e-57,-56.547135
6,આ ટ્રેનિંગ કેન્દ્રસરકારના આરોગ્ય કર્મચારી અને ...,0.000000e+00,-116.997424
7,વહેલી સવારથી જ ઠંડા પવન ફ્ૂંકાતા રહીશો મુશ્કેલ...,0.000000e+00,-39.016748
8,"દર્શનાબેન પંડયા, પ્રીતીબેન દોશી, પીનાબેન કોટક,...",1.033839e-41,-40.985547
9,સેન્ટ્રિપ્ટલ કારકિર્દીની રેખા (નિયંત્રણ કેન્દ્...,0.000000e+00,-66.936696


# 3rd

# build probability dictionaries 

In [40]:
from collections import defaultdict

def build_probabilities_fast(n_gram_counts):
    probs = {}
    
    # Unigram
    if all(isinstance(k, str) for k in n_gram_counts.keys()):
        total = sum(n_gram_counts.values())
        probs = {k: v/total for k, v in n_gram_counts.items()}
        return probs
    
    # Bigram, trigram, quadrigram
    prefix_totals = defaultdict(int)
    for k, v in n_gram_counts.items():
        prefix = k[:-1]
        prefix_totals[prefix] += v

    # Now compute probabilities in one pass
    probs = {k: v / prefix_totals[k[:-1]] for k, v in n_gram_counts.items()}
    return probs

# Build probabilities
uni_probs = build_probabilities_fast(uni_counts)
print("unigram MLE done ")
bi_probs = build_probabilities_fast(bi_counts)
print("bigram MLE done ")
tri_probs = build_probabilities_fast(tri_counts)
print("trigram MLE done")
quad_probs = build_probabilities_fast(quad_counts)
print("quadrigram MLW done")

unigram MLE done 
bigram MLE done 
trigram MLE done
quadrigram MLW done


In [41]:
def save_probs_pickle(probs, filename):
    with open(filename, "wb") as f:
        pickle.dump(probs, f)

# Save all probabilities
save_probs_pickle(uni_probs, "uni_probs_MLE.pkl")
save_probs_pickle(bi_probs, "bi_probs_MLE.pkl")
save_probs_pickle(tri_probs, "tri_probs_MLE.pkl")
save_probs_pickle(quad_probs, "quad_probs_MLE.pkl")


# preindexing 

In [55]:
from collections import defaultdict
import random

# Pre-index n-grams by prefix
def index_ngrams_by_prefix(ngram_probs):
    prefix_index = defaultdict(dict)
    for k, prob in ngram_probs.items():
        if isinstance(k, tuple):
            prefix = k[:-1]
            word = k[-1]
        else:  # unigram
            prefix = ()
            word = k
        prefix_index[prefix][word] = prob
    return prefix_index


# greedy sentence generation

In [56]:
def generate_sentence_greedy_fast(n, prefix_index, max_len=15):
    sentence = []

    # Start with random n-gram
    if n == 1:
        sentence.append(random.choices(list(prefix_index[()].keys()), 
                                       weights=prefix_index[()].values())[0])
    else:
        # pick random starting prefix
        start_prefix = random.choice(list(prefix_index.keys()))
        if start_prefix != ():
            sentence.extend(list(start_prefix))

    for _ in range(max_len):
        prefix = tuple(sentence[-(n-1):]) if n > 1 else ()
        candidates = prefix_index.get(prefix, None)
        if not candidates:
            break
        word = max(candidates, key=candidates.get)  # greedy
        sentence.append(word)

    return ' '.join(sentence)


# beam search 

In [57]:
def generate_sentence_beam_fast(n, prefix_index, beam_width=20, max_len=15):
    beams = [([], 1.0)]

    for _ in range(max_len):
        new_beams = []
        for sentence, prob in beams:
            prefix = tuple(sentence[-(n-1):]) if n > 1 else ()
            candidates = prefix_index.get(prefix, None)
            if not candidates:
                new_beams.append((sentence, prob))
                continue
            for word, p in candidates.items():
                new_beams.append((sentence + [word], prob*p))
        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]

    return ' '.join(beams[0][0])


In [58]:
def generate_and_save_sentences(n, ngram_probs, filename_prefix):
    greedy_sentences = [generate_sentence_greedy(n, ngram_probs) for _ in range(100)]
    beam_sentences = [generate_sentence_beam(n, ngram_probs) for _ in range(100)]

    # Show 5 examples
    print(f"Greedy examples for {n}-gram:")
    for s in greedy_sentences[:5]:
        print(s)
    print(f"\nBeam Search examples for {n}-gram:")
    for s in beam_sentences[:5]:
        print(s)

    # Save to disk
    with open(f"greedy_{filename_prefix}.txt", "w") as f:
        f.write('\n'.join(greedy_sentences))
    with open(f"beam_{filename_prefix}.txt", "w") as f:
        f.write('\n'.join(beam_sentences))


In [ ]:
generate_and_save_sentences(1, uni_probs, "unigram")
print("unigram done ")
generate_and_save_sentences(2, bi_probs, "bigram")
print("bigram done")
generate_and_save_sentences(3, tri_probs, "trigram")
print("trigra done ")
generate_and_save_sentences(4, quad_probs, "quadrigram")
print("quadrigram done")


unigram done<br>
bigram done<br>
trigra done<br>
qudrigram done<br>